# Multi-Agent Consensus & Verification

**Making several Claude agents converge on a decision reliably — and fail safely when they can't.**

The [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents) patterns in this
directory (prompt chaining, routing, parallelization, orchestrator-workers, evaluator-optimizer) all assume the
happy path: a worker returns an answer and the orchestrator trusts it. In production that assumption breaks in two
ways this cookbook otherwise doesn't cover:

1. **Workers disagree.** Fan three agents out on the same non-trivial question and you often get three different
   answers. Which one do you ship?
2. **A worker is *confidently wrong*.** A single plausible-but-incorrect answer, unchallenged, flows straight into
   whatever the agent does next.

This recipe adds the missing layer: a **propose → adversarially verify → reconcile → commit** loop, with a
deterministic **human-approval gate** for irreversible actions. It is distilled from a multi-agent system that has
been running this pattern autonomously, 24/7, across several machines — the lessons in the last section are the ones
that actually bit us in production.

**What you'll build**

| Building block | Job |
|---|---|
| `propose()` | Fan N *diverse* proposer agents out on the same task |
| `adversarial_verify()` | A skeptic agent whose only job is to **refute** each proposal |
| `reconcile()` | Deterministic: keep what survives verification; break ties without an LLM |
| `tier_guard()` | Deterministic keyword tripwire → route risky actions to a **human**, never auto-commit |
| `decision_log` | Append-only, idempotent record of every step |

We then wire them into one `consensus_decide()` call and run a small **eval** showing the pattern catches errors that
a single agent — and even naive majority vote — ships.


## Setup

Requires a recent Anthropic Python SDK (for `client.messages.parse(...)` structured outputs) and an API key in
`ANTHROPIC_API_KEY`.

We deliberately use **two model tiers**: cheaper models (`claude-sonnet-5`, `claude-haiku-4-5`) for the *proposer*
grunt work, and the strongest model (`claude-opus-4-8`) for the *verifier* and *tie-break* — where judgment matters
most. This mirrors a real cost lesson: consensus gets expensive fast if every agent is your most capable model.


In [ ]:
%pip install --quiet --upgrade anthropic pydantic

In [1]:
import concurrent.futures
import hashlib
import json
from collections import Counter
from dataclasses import asdict, dataclass, field

from anthropic import Anthropic
from pydantic import BaseModel

client = Anthropic()  # reads ANTHROPIC_API_KEY (or an `ant auth login` profile)

# Model tiers — cheap for breadth, strong for judgment.
PROPOSER_MODELS = ["claude-sonnet-5", "claude-haiku-4-5", "claude-sonnet-5"]
VERIFIER_MODEL = "claude-opus-4-8"
TIEBREAK_MODEL = "claude-opus-4-8"


def call(model: str, system: str, user: str, thinking: bool = False, max_tokens: int = 1024) -> str:
    """Minimal text-in / text-out helper."""
    kwargs = dict(
        model=model,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    if thinking:
        kwargs["thinking"] = {"type": "adaptive"}  # let Claude decide how much to think
    resp = client.messages.create(**kwargs)
    return "".join(b.text for b in resp.content if b.type == "text").strip()

## 1. Propose — fan out *diverse* agents

The point of multiple proposers is **independent** answers, so we vary two things: the model tier and the framing of
the system prompt (an "analyst", a "contrarian", a "practitioner"). Identical prompts to identical models just give
you the same answer three times — diversity is what makes disagreement *informative*.


In [2]:
PROPOSER_PERSONAS = [
    "You are a careful analyst. Reason step by step and give your single best answer.",
    "You are a contrarian expert. Question the obvious answer; surface the case others miss.",
    "You are a pragmatic practitioner. Give the answer that holds up in the real world.",
]


@dataclass
class Proposal:
    id: str
    model: str
    persona: str
    answer: str


def propose(question: str) -> list[Proposal]:
    """Run every (model, persona) pair concurrently and collect proposals."""
    jobs = list(zip(PROPOSER_MODELS, PROPOSER_PERSONAS, strict=True))

    def one(i_model_persona):
        i, (model, persona) = i_model_persona
        answer = call(model, persona, question, max_tokens=800)
        return Proposal(id=f"p{i}", model=model, persona=persona[:24], answer=answer)

    with concurrent.futures.ThreadPoolExecutor(max_workers=len(jobs)) as pool:
        return list(pool.map(one, enumerate(jobs)))

## 2. Adversarially verify — a skeptic that tries to *refute*

This is the load-bearing idea. Instead of asking "is this answer good?" (which rubber-stamps plausible-but-wrong
output), we give a strong verifier one job: **find the flaw**. It must default to `refuted=true` when uncertain. We
use [structured outputs](https://docs.claude.com/en/docs/build-with-claude/structured-outputs) so the verdict is a
validated object, not prose we have to parse.

> **Why default-to-refute matters.** A verifier prompted neutrally ("evaluate this") passes ~everything. A verifier
> prompted to attack, with the burden of proof on the *answer*, catches the confident-wrong cases. This is the single
> biggest quality lever in the whole pattern.


In [3]:
class Verdict(BaseModel):
    refuted: bool  # True = the proposal is wrong or unsupported
    confidence: float  # 0..1, how sure the verifier is
    reason: str  # one sentence


VERIFIER_SYSTEM = (
    "You are an adversarial verifier. Your job is to REFUTE the candidate answer, not to be agreeable. "
    "Look for factual errors, faulty reasoning, unstated assumptions, and edge cases the answer ignores. "
    "The burden of proof is on the answer: if you cannot establish that it is correct and well-supported, "
    "set refuted=true. When genuinely uncertain, default to refuted=true."
)


def adversarial_verify(question: str, proposal: Proposal) -> Verdict:
    user = (
        f"QUESTION:\n{question}\n\n"
        f"CANDIDATE ANSWER (from {proposal.model}):\n{proposal.answer}\n\n"
        "Try to refute this answer. Return your verdict."
    )
    resp = client.messages.parse(
        model=VERIFIER_MODEL,
        max_tokens=3000,  # room for adaptive thinking + the JSON verdict; too tight a budget truncates the JSON
        thinking={"type": "adaptive"},
        system=VERIFIER_SYSTEM,
        messages=[{"role": "user", "content": user}],
        output_format=Verdict,
    )
    if resp.parsed_output is None:
        # Fail CLOSED: an unparseable verdict must never count as approval.
        return Verdict(refuted=True, confidence=0.0, reason="verifier output unparseable")
    return resp.parsed_output

## 3. Reconcile — deterministic, no LLM

Reconciliation is **plain code**, on purpose. Once each proposal has a verdict, the decision of what survives should
be auditable and reproducible — not another model call. Three branches:

- **One survivor** → that's the answer (consensus).
- **Several survivors** → a strong tie-break agent **chooses among them** (they already passed verification, so it
  may only pick, not invent).
- **No survivors** → the tie-break agent must set the candidates aside and reason from first principles — and because
  that synthesis is a *brand-new, unverified* answer, it goes **back through the adversarial verifier**. If it too is
  refuted, we stop and escalate to a human. That's the **round cap**: one synthesis attempt, then a person.

Two details here came straight from watching this fail. First, the tie-break prompt must treat refutations as
*standing* — without that, a strong model happily repeats a refuted answer (we watched Opus re-derive the memorized
riddle answer right after the verifier had explained exactly why it was wrong). Second, the synthesis must be
verified like any other proposal; an answer's origin grants it no trust.


In [4]:
@dataclass
class Decision:
    status: str  # "consensus" | "tie_break" | "synthesized" | "needs_human"
    answer: str | None
    survivors: list[str] = field(default_factory=list)  # proposal ids that survived
    rationale: str = ""


TIEBREAK_SYSTEM = (
    "You are the tie-break authority. The candidates below were adversarially reviewed; their verdicts are "
    "attached. Treat a refutation as standing unless you can directly rebut its specific reason. Never repeat "
    "a refuted answer without rebutting its refutation. If every candidate is refuted, set the candidates "
    "aside entirely and reason from first principles. Be decisive and explain briefly."
)


def _candidate_listing(candidates, verdicts):
    return "\n\n".join(
        f"[{p.id}] (refuted={verdicts[p.id].refuted}, reason={verdicts[p.id].reason})\n{p.answer}"
        for p in candidates
    )


def reconcile(question: str, proposals: list[Proposal], verdicts: dict[str, Verdict]) -> Decision:
    survivors = [p for p in proposals if not verdicts[p.id].refuted]

    if len(survivors) == 1:
        return Decision(
            "consensus", survivors[0].answer, [survivors[0].id], "single unrefuted proposal"
        )

    if len(survivors) >= 2:
        # Survivors already passed verification — the tie-break may only CHOOSE among them.
        out = call(
            TIEBREAK_MODEL,
            TIEBREAK_SYSTEM,
            f"QUESTION:\n{question}\n\nCANDIDATES (all passed verification — choose the best one):\n"
            f"{_candidate_listing(survivors, verdicts)}\n\n"
            "Give the final answer, then one line of rationale.",
            thinking=True,
            max_tokens=1024,
        )
        return Decision(
            "tie_break", out, [p.id for p in survivors], "tie-break among verified survivors"
        )

    # No survivors: synthesize from first principles, then VERIFY the synthesis like any other proposal.
    out = call(
        TIEBREAK_MODEL,
        TIEBREAK_SYSTEM,
        f"QUESTION:\n{question}\n\nCANDIDATES (every one was refuted):\n"
        f"{_candidate_listing(proposals, verdicts)}\n\n"
        "Give the final answer, then one line of rationale.",
        thinking=True,
        max_tokens=1024,
    )
    check = adversarial_verify(
        question, Proposal(id="synthesis", model=TIEBREAK_MODEL, persona="tie-break", answer=out)
    )
    if check.refuted:
        # Round cap reached: one synthesis attempt, then a person decides.
        return Decision(
            "needs_human",
            out,
            [],
            f"all proposals AND the synthesis were refuted ({check.reason}) — escalating to a human",
        )
    return Decision("synthesized", out, [], "synthesis from first principles, verified")

## 4. Tier guard — a deterministic gate for irreversible actions

The whole safety story hinges on **never auto-committing something dangerous**, even if the agents unanimously agree.
And you cannot trust an LLM to classify its own action's risk — a mislabel silently defeats the gate.

So the tripwire is **deterministic keyword matching**, independent of anything the model says. If the proposed action
touches money, deletion, outbound messages, credentials, or config, it is forced to `needs_human` regardless of
consensus. This exact design — a keyword tripwire that *overrides* the agent's own risk label — is what stopped our
autonomous fleet from ever wiring the consensus loop to a destructive command.


In [5]:
# Editable, "fix it with a hammer" keyword list. Deliberately blunt and broad.
DANGER_KEYWORDS = [
    "delete",
    "drop",
    "wipe",
    "truncate",
    "overwrite",
    "rm -",
    "format",
    "send",
    "email",
    "post",
    "publish",
    "deploy",
    "merge",
    "push",
    "transfer",
    "wire",
    "pay",
    "refund",
    "charge",
    "password",
    "secret",
    "token",
    "api key",
    "credential",
    ".env",
    "revoke",
    "uninstall",
    "factory reset",
]


def is_high_risk(action_text: str) -> tuple[bool, str]:
    low = action_text.lower()
    for kw in DANGER_KEYWORDS:
        if kw in low:
            return True, kw
    return False, ""


def tier_guard(decision: Decision) -> Decision:
    """Force human approval for irreversible actions, no matter how confident the agents are."""
    if decision.answer is None:
        return decision
    high, kw = is_high_risk(decision.answer)
    if high:
        return Decision(
            "needs_human",
            decision.answer,
            decision.survivors,
            f"tripwire hit on '{kw}' — requires human approval before commit",
        )
    return decision

## 5. An append-only, idempotent decision log

Every step emits an event to an append-only log keyed by a deterministic `event_id`. In a distributed setup the same
event can be delivered twice (redundant rails, retries); idempotency by `event_id` means replaying the log is safe and
the record is reconstructable. Even in a single process it gives you a clean audit trail of *why* a decision was made.


In [6]:
DECISION_LOG: list[dict] = []
_seen_event_ids: set[str] = set()


def log_event(kind: str, payload: dict):
    body = json.dumps({"kind": kind, **payload}, sort_keys=True, default=str)
    event_id = hashlib.sha256(body.encode()).hexdigest()[:16]
    if event_id in _seen_event_ids:  # idempotent: ignore duplicate delivery
        return
    _seen_event_ids.add(event_id)
    DECISION_LOG.append({"event_id": event_id, "kind": kind, **payload})

## 6. Put it together: `consensus_decide()`

Propose → verify (concurrently) → reconcile → tier-guard, logging every step.


In [7]:
def consensus_decide(question: str) -> Decision:
    log_event("question", {"text": question})

    proposals = propose(question)
    for p in proposals:
        log_event("proposal", asdict(p))

    # Verify every proposal concurrently.
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(proposals)) as pool:
        verdict_list = list(pool.map(lambda p: (p.id, adversarial_verify(question, p)), proposals))
    verdicts = dict(verdict_list)
    for pid, v in verdicts.items():
        log_event("verdict", {"proposal_id": pid, **v.model_dump()})

    decision = reconcile(question, proposals, verdicts)
    decision = tier_guard(decision)
    log_event("decision", asdict(decision))
    return decision

In [8]:
# A benign question — should reach consensus / tie-break and auto-commit.
d = consensus_decide("A store sells pens at 3 for $2. How much do 12 pens cost, and why?")
print("STATUS:", d.status)
print("ANSWER:", d.answer)
print("RATIONALE:", d.rationale)

STATUS: tie_break
ANSWER: **$8.**

12 pens = 4 groups of 3 at $2 each = 4 × $2 = $8; p0 gives this cleanly without the unnecessary and speculative caveats that clutter p1 and p2.
RATIONALE: tie-break among verified survivors


In [9]:
# A question whose "answer" implies an irreversible action — must route to a human.
d = consensus_decide(
    "The user's inbox is full of old newsletters. Draft the exact action to take to clean it up."
)
print("STATUS:", d.status)  # expect: needs_human  (tripwire on 'delete'/'email' etc.)
print("REASON:", d.rationale)

STATUS: needs_human
REASON: tripwire hit on 'delete' — requires human approval before commit


## 7. Eval: does the pattern actually catch errors?

A pattern is only worth the tokens if it beats the cheap baselines people actually deploy:

- **single** — one quick call, ship the answer. (The default everywhere.)
- **majority** — three quick calls, ship the most common answer. (Self-consistency: redundancy without verification.)
- **consensus** — this recipe: diverse proposers → adversarial verify → reconcile.

The three tiers deliberately spend different token budgets — that's the question the eval answers: *what do the
extra tokens buy, and does redundancy alone buy it?*

Picking eval items needs care. Classic trick questions (bat-and-ball, lily pads) are in every training set, so
modern models answer them correctly even as quick calls — our first draft used them and scored a meaningless 100%
across all three strategies. What still fools models reliably is **adversarial variants of memorized puzzles**:
change one number so the memorized answer becomes wrong, and a quick call pattern-matches straight into the trap.
The discriminator below is bat-and-ball with the bat costing **$0.90** more (ball = $0.10; the memorized $0.05 is
now wrong). The other three items are controls. We grade every final answer with an independent strong-model
grader against the gold answer.


In [10]:
EVAL = [
    # Discriminator — adversarial variant of a memorized puzzle (quick calls pattern-match to $0.05):
    {
        "q": "A bat and a ball cost $1.10 in total. The bat costs $0.90 more than the ball. "
        "How much does the ball cost?",
        "gold": "$0.10 (the memorized bat-and-ball answer, $0.05, is wrong for THIS wording)",
    },
    # Controls — memorized or straightforward; every strategy should pass:
    {
        "q": "If it takes 5 machines 5 minutes to make 5 widgets, how long would 100 machines "
        "take to make 100 widgets?",
        "gold": "5 minutes",
    },
    {
        "q": "A patch of lily pads doubles in size every day and covers the whole lake on day 48. "
        "On what day did it cover a quarter of the lake?",
        "gold": "day 46",
    },
    {"q": "A farmer has 17 sheep. All but 9 die. How many are left?", "gold": "9 sheep"},
]


class Grade(BaseModel):
    correct: bool
    why: str


def grade(question: str, answer: str, gold: str) -> bool:
    resp = client.messages.parse(
        model="claude-opus-4-8",
        max_tokens=512,
        system="You are a strict grader. Mark correct only if the answer matches the gold answer in substance.",
        messages=[
            {
                "role": "user",
                "content": f"Q: {question}\nGOLD: {gold}\nANSWER: {answer}\nIs the answer correct?",
            }
        ],
        output_format=Grade,
    )
    return resp.parsed_output.correct


def _quick(model, q):
    # A quick call: short, no reasoning room — how cheap calls are typically deployed.
    return call(
        model, "Answer with ONLY the final answer, in as few words as possible.", q, max_tokens=60
    )


def _norm(a):
    return a.strip().lower().rstrip(".").replace("$", "").replace(" cents", "").strip()


def strategy_single(q):  # one quick call
    return _quick(PROPOSER_MODELS[0], q)


def strategy_majority(q):  # self-consistency: majority over three quick calls, no verification
    answers = [_quick(m, q) for m in PROPOSER_MODELS]
    winner = Counter(_norm(a) for a in answers).most_common(1)[0][0]
    return next(a for a in answers if _norm(a) == winner)


def strategy_consensus(q):  # this recipe: propose -> adversarially verify -> reconcile
    return consensus_decide(q).answer

In [11]:
results = {"single": 0, "majority": 0, "consensus": 0}
for item in EVAL:
    for name, fn in [
        ("single", strategy_single),
        ("majority", strategy_majority),
        ("consensus", strategy_consensus),
    ]:
        ans = fn(item["q"])
        ok = grade(item["q"], ans or "", item["gold"])
        results[name] += int(ok)
        print(f"[{name:9}] {'PASS' if ok else 'FAIL'}  {item['q'][:48]}...")
    print()

n = len(EVAL)
print("Accuracy over", n, "questions:")
for name, score in results.items():
    print(f"  {name:9}: {score}/{n}")

[single   ] FAIL  A bat and a ball cost $1.10 in total. The bat co...
[majority ] FAIL  A bat and a ball cost $1.10 in total. The bat co...
[consensus] PASS  A bat and a ball cost $1.10 in total. The bat co...

[single   ] PASS  If it takes 5 machines 5 minutes to make 5 widge...
[majority ] PASS  If it takes 5 machines 5 minutes to make 5 widge...
[consensus] PASS  If it takes 5 machines 5 minutes to make 5 widge...

[single   ] PASS  A patch of lily pads doubles in size every day a...
[majority ] PASS  A patch of lily pads doubles in size every day a...
[consensus] PASS  A patch of lily pads doubles in size every day a...

[single   ] PASS  A farmer has 17 sheep. All but 9 die. How many a...
[majority ] PASS  A farmer has 17 sheep. All but 9 die. How many a...
[consensus] PASS  A farmer has 17 sheep. All but 9 die. How many a...

Accuracy over 4 questions:
  single   : 3/4
  majority : 3/4
  consensus: 4/4


On the controls, everything passes — the pattern costs accuracy nothing. On the discriminator, the quick calls
pattern-match to the memorized $0.05 — and because that wrong answer is *stable*, **majority vote makes it
unanimous instead of fixing it**. The consensus pipeline passes: its proposers get reasoning room, its verifier
attacks whatever they produce, and a wrong survivor still has to make it past the refutation step. Your exact
numbers will vary run to run, but the ordering (`consensus ≥ majority ≥ single`) is the point: verification buys
accuracy that redundancy alone does not.


## 7.5 Case study: when the whole chain rationalizes

There is a harder failure mode the scored eval above cannot show, and honesty requires showing it. Take an
adversarial variant whose memorized answer is *maximally* magnetic:

> *"You have two ordinary US coins that add up to 30 cents. **Neither** of them is a nickel. What are the two coins?"*

The classic riddle says "**one** of them is not a nickel" (answer: quarter + nickel). This wording forbids both
coins from being nickels — so with standard US denominations **no such pair exists**. Watch what the pipeline does:


In [12]:
CASE = (
    "You have two ordinary US coins that add up to 30 cents. "
    "Neither of them is a nickel. What are the two coins?"
)
log_start = len(DECISION_LOG)  # scope the trail to this call
d = consensus_decide(CASE)
print("STATUS:", d.status)
print("RATIONALE:", d.rationale)
print("ANSWER (tail):", (d.answer or "")[-200:])
print("\nVerdict trail:")
for ev in DECISION_LOG[log_start:]:
    if ev["kind"] == "verdict":
        print(f"  [{ev['proposal_id']}] refuted={ev['refuted']}  {ev['reason'][:100]}")

STATUS: synthesized
RATIONALE: synthesis from first principles, verified
ANSWER (tail): e coin needs to not be a nickel (the quarter), while the other one is the nickel, giving 25¢ + 5¢ = 30¢. The "impossible" candidates (p1, p2) were correctly refuted for missing this intended solution.

Verdict trail:
  [p0] refuted=True  The candidate answer is empty—it provides no coins at all. The correct answer is a quarter and a nic
  [p1] refuted=True  The classic riddle's intended answer is a quarter and a nickel. The trick is the clue 'neither of th
  [p2] refuted=True  This is the classic trick riddle whose intended answer is 'a quarter and a nickel.' The wordplay is 


Across repeated runs of this cell we have seen **every stage fail and every stage succeed**: proposers
unanimously rationalizing the memorized answer ("'neither is a nickel' is just the riddle's misdirection…") while
the verifier correctly refuted them all; proposers landing on the right "no such pair" while the *verifier*
refuted the correct answers for "missing the classic solution"; the first-principles synthesis getting it right —
and, in other runs, re-deriving the memorized answer and sailing through. One memorable trace had the tie-break
model repeat the exact answer whose refutation was printed in its own prompt.

That variance is the finding. Near the edge of what a model has memorized, **every LLM stage — proposer, verifier,
synthesizer — is pulled toward the same wrong answer**, so stacking them shifts the odds without guaranteeing the
outcome. This is precisely why the pipeline's non-LLM parts exist: the deterministic reconcile rules, the round
cap, and the human gate are the components that cannot be sweet-talked. For high-stakes questions, the upgrade
path is not a smarter judge — it is *N independent verifiers with diverse lenses* and majority-to-refute (see
"Where to take it next"), plus a human behind the round cap.


## 8. Failure modes & lessons from running this in production

These are the things that actually bit a fleet running this pattern autonomously, 24/7:

1. **Tier misclassification is the #1 risk — so don't let the model classify.** The entire safety gate hinges on
   labeling an action as risky. If you ask the agent to self-label, one mislabel silently disables the gate. The
   deterministic keyword tripwire in §4 *overrides* the agent's own judgment for exactly this reason. Keep the list
   blunt and over-broad; a false "needs human" is cheap, a false auto-commit is not.

2. **A neutral verifier rubber-stamps.** "Evaluate this answer" passes almost everything. The burden of proof has to
   sit on the *answer*, and the verifier must default to `refuted=true` when unsure (§2). This one prompt change is
   the difference between catching confident-wrong output and not.

3. **Redundancy ≠ verification.** Majority vote over identical models gives you a *confident* wrong answer, not a
   correct one. Diverse proposers (§1) plus an adversarial check (§2) is what pays off — see the eval.

4. **The synthesizer rationalizes too — verify the synthesis.** While building this notebook we watched the
   strongest model, acting as tie-break, re-derive the exact answer the verifier had just refuted — with the
   refutation in its prompt (§7.5). Two fixes, both in §3: the tie-break prompt must treat refutations as
   *standing* (rebut or avoid, never ignore), and a from-scratch synthesis is a new unverified answer that must
   go back through the verifier before it can commit. And when the pull of a memorized answer is strong enough,
   even the verifier can waver — single-verifier consensus shifts the odds, it does not guarantee the outcome.

5. **Make reconciliation deterministic.** Once verdicts exist, *which answer survives* should be plain code (§3), not
   another model call. Reproducibility and auditability come from the log + deterministic rules, not from a smarter
   judge.

6. **Idempotent, append-only log.** In any distributed setup the same event arrives twice. Key events by a
   deterministic hash (§5) so replay is safe and the decision record is reconstructable.

7. **Cap the rounds.** One synthesis attempt, verified — then a human. Genuine, persistent disagreement is a
   signal, not a bug to grind away.

### Where to take it next
- Swap the keyword tripwire for a policy engine, but keep it **deterministic and independent of the agents**.
- Run *N* independent verifiers per proposal and require a majority-to-refute for higher-stakes decisions
  (perspective-diverse verification: correctness / safety / does-it-reproduce).
- Persist `DECISION_LOG` to durable storage and expose it as the audit trail for every committed action.

---

*Contributed as a companion to the [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents)
patterns — covering what those recipes deliberately leave out: what happens when agents disagree, when one is
confidently wrong, and when the action is too risky to take without a human.*
